# Real-Time Crisis Detection System
## Notebook 01 — Data Ingestion
**Goal:** Collect raw data from Twitter, Reddit, News RSS, BMKG, and PetaBencana. Normalize to unified schema, deduplicate, and save to Google Drive.

In [1]:
# Environment Setup
import os, sys, warnings, logging, random
import numpy as np
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
random.seed(42); np.random.seed(42)

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_URL_BASE' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/10Academy/crisis-detection-system'
elif IN_KAGGLE:
    PROJECT_DIR = '/kaggle/working/crisis-detection-system'
else:
    PROJECT_DIR = os.path.abspath('..')

DATA_RAW        = f'{PROJECT_DIR}/data/raw'
DATA_PROCESSED  = f'{PROJECT_DIR}/data/processed'
DATA_EXTERNAL   = f'{PROJECT_DIR}/data/external'
MODELS_DIR      = f'{PROJECT_DIR}/models'
OUTPUTS_DIR     = f'{PROJECT_DIR}/outputs'
FIGURES_DIR     = f'{PROJECT_DIR}/outputs/figures'
CONFIG_DIR      = f'{PROJECT_DIR}/config'
UTILS_DIR       = f'{PROJECT_DIR}/utils'

for d in [DATA_RAW, DATA_PROCESSED, DATA_EXTERNAL, MODELS_DIR,
          OUTPUTS_DIR, FIGURES_DIR, CONFIG_DIR, UTILS_DIR]:
    os.makedirs(d, exist_ok=True)

if UTILS_DIR not in sys.path:
    sys.path.insert(0, UTILS_DIR)

print(f' Project: {PROJECT_DIR}')
print(f' Env: {"Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "Local"}')

Mounted at /content/drive
✅ Project: /content/drive/MyDrive/10Academy/crisis-detection-system
✅ Env: Colab


In [2]:
# Install dependencies
!pip install -q tweepy feedparser requests pyyaml tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.8 MB/s eta 0:00:00


In [4]:
# ============================================================
# CELL: Auto-Fix Directory Structure & Move Files
# ============================================================
import os
import shutil

# Define the base project directory
BASE_DIR = '/content/drive/MyDrive/10Academy/crisis-detection-system'
UTILS_DIR = os.path.join(BASE_DIR, 'utils')
CONFIG_DIR = os.path.join(BASE_DIR, 'config')

# Make sure the target folders exist
os.makedirs(UTILS_DIR, exist_ok=True)
os.makedirs(CONFIG_DIR, exist_ok=True)

# 1. Places we should look for misplaced files
search_paths = [
    '/content', # Root Colab directory (where uploaded files usually land)
    CONFIG_DIR  # In case you accidentally moved .py files to the config folder
]

moved_py = 0
moved_yaml = 0

print(" Scanning for misplaced files...")

for search_path in search_paths:
    if not os.path.exists(search_path):
        continue

    for filename in os.listdir(search_path):
        source_path = os.path.join(search_path, filename)

# Skip directories
        if os.path.isdir(source_path):
            continue

# Move Python files to the utils/ folder
        if filename.endswith('.py'):
            dest_path = os.path.join(UTILS_DIR, filename)
# Only move if the source and destination are different
            if source_path != dest_path:
                shutil.move(source_path, dest_path)
                print(f" Moved utility script: {filename} -> {UTILS_DIR}")
                moved_py += 1

# Move Configuration files to the config/ folder
        elif filename.endswith('.yaml'):
            dest_path = os.path.join(CONFIG_DIR, filename)
            if source_path != dest_path:
                shutil.move(source_path, dest_path)
                print(f" Moved config file: {filename} -> {CONFIG_DIR}")
                moved_yaml += 1

print("-" * 40)
print(f" Cleanup complete! Moved {moved_py} .py files and {moved_yaml} .yaml files.")
print("You can now safely delete this cell and re-run your notebook from Cell 1.")

🔍 Scanning for misplaced files...
✅ Moved config file: model_config.yaml -> /content/drive/MyDrive/10Academy/crisis-detection-system/config
✅ Moved utility script: llm_utils.py -> /content/drive/MyDrive/10Academy/crisis-detection-system/utils
✅ Moved utility script: clustering_utils.py -> /content/drive/MyDrive/10Academy/crisis-detection-system/utils
✅ Moved utility script: alert_utils.py -> /content/drive/MyDrive/10Academy/crisis-detection-system/utils
✅ Moved utility script: geo_utils.py -> /content/drive/MyDrive/10Academy/crisis-detection-system/utils
✅ Moved config file: paths.yaml -> /content/drive/MyDrive/10Academy/crisis-detection-system/config
✅ Moved utility script: temporal_utils.py -> /content/drive/MyDrive/10Academy/crisis-detection-system/utils
✅ Moved config file: api_keys.yaml -> /content/drive/MyDrive/10Academy/crisis-detection-system/config
✅ Moved utility script: ingestion_utils.py -> /content/drive/MyDrive/10Academy/crisis-detection-system/utils
✅ Moved utility scrip

In [5]:
# Load API credentials + define missing ingestion functions
import pandas as pd, yaml, re, requests, feedparser
from datetime import datetime, timedelta, timezone
from tqdm import tqdm

# Load API keys
try:
    with open(f'{CONFIG_DIR}/api_keys.yaml') as f:
        keys = yaml.safe_load(f)
    TWITTER_BEARER = keys.get('twitter', {}).get('bearer_token', '')
    GEMINI_KEY     = keys.get('llm', {}).get('api_key', '')
    print(' API keys loaded.')
except Exception as e:
    print(f'  {e}  →  running in sample mode.')
    TWITTER_BEARER = ''
    GEMINI_KEY     = ''

# Import what's available, define what's missing
try:
    from ingestion_utils import fetch_tweets
except ImportError:
    def fetch_tweets(*args, **kwargs): return pd.DataFrame()

try:
    from ingestion_utils import fetch_reddit_posts
except ImportError:
    def fetch_reddit_posts(keywords, limit=50):
        rows = []
        for kw in keywords[:3]:
            query = kw.replace(' ', '+')
            try:
                feed = feedparser.parse(f'https://www.reddit.com/search.rss?q={query}&sort=new')
                for e in feed.entries[:limit//len(keywords)]:
                    summary = re.sub(r'<[^>]+>', ' ', getattr(e,'summary',''))
                    rows.append({'post_id':f"RD_{abs(hash(e.id))}",'raw_text':(e.title+' '+summary).strip(),
                                 'timestamp_raw':e.get('published',''),'user_id':None,'location_raw':None,
                                 'geo_lat':None,'geo_lon':None,'media_present':False,
                                 'engagement_score':0,'source_platform':'reddit','label':None})
            except: pass
        return pd.DataFrame(rows)

try:
    from ingestion_utils import fetch_rss_entries
except ImportError:
    def fetch_rss_entries(feed_urls=None, keywords=None):
        if feed_urls is None: feed_urls=['https://www.antaranews.com/rss/terkini.xml']
        if keywords is None:  keywords=['banjir','gempa','bencana']
        rows = []
        for url in feed_urls:
            try:
                feed = feedparser.parse(url)
                for e in feed.entries:
                    text = (e.title+' '+getattr(e,'summary','')).lower()
                    if any(kw in text for kw in keywords):
                        rows.append({'post_id':f"NEWS_{abs(hash(e.get('id',e.link)))}",
                                     'raw_text':(e.title+'. '+re.sub(r'<[^>]+>',' ',getattr(e,'summary',''))).strip(),
                                     'timestamp_raw':str(e.get('published','')),'user_id':None,
                                     'location_raw':None,'geo_lat':None,'geo_lon':None,
                                     'media_present':False,'engagement_score':0,
                                     'source_platform':'news_rss','label':None})
            except: pass
        return pd.DataFrame(rows)

# Define fetch_bmkg_alerts (new function)
def fetch_bmkg_alerts(min_magnitude=4.0):
    """Fetch earthquake alerts from BMKG public API."""
    print('Fetching BMKG alerts...')
    rows = []
    try:
        r = requests.get('https://data.bmkg.go.id/DataMKG/TEWS/autogempa.json', timeout=10)
        eq = r.json().get('Infogempa',{}).get('gempa',{})
        if eq and float(eq.get('Magnitude',0)) >= min_magnitude:
            rows.append({'post_id':f"BMKG_{eq.get('DateTime','').replace(' ','_')}",
                         'raw_text':f"Earthquake M{eq.get('Magnitude')} depth {eq.get('Kedalaman')} in {eq.get('Wilayah')}. {eq.get('Potensi','')}",
                         'timestamp_raw':eq.get('DateTime',''),'user_id':'BMKG_official',
                         'location_raw':eq.get('Wilayah',''),'geo_lat':None,'geo_lon':None,
                         'media_present':False,'engagement_score':100,'source_platform':'bmkg','label':'earthquake'})
    except Exception as e: print(f'  BMKG latest: {e}')
    try:
        r2 = requests.get('https://data.bmkg.go.id/DataMKG/TEWS/gempaterkini.json', timeout=10)
        for eq in r2.json().get('Infogempa',{}).get('gempa',[]):
            if float(eq.get('Magnitude',0)) >= min_magnitude:
                rows.append({'post_id':f"BMKG_{eq.get('DateTime','').replace(' ','_')}_{len(rows)}",
                             'raw_text':f"Earthquake M{eq.get('Magnitude')} at depth {eq.get('Kedalaman')} in {eq.get('Wilayah')}.",
                             'timestamp_raw':eq.get('DateTime',''),'user_id':'BMKG_official',
                             'location_raw':eq.get('Wilayah',''),'geo_lat':None,'geo_lon':None,
                             'media_present':False,'engagement_score':100,'source_platform':'bmkg','label':'earthquake'})
    except Exception as e: print(f'  BMKG list: {e}')
    print(f' BMKG: {len(rows)} alerts')
    return pd.DataFrame(rows)

# Define fetch_petabencana_reports (new function)
def fetch_petabencana_reports():
    """Fetch flood reports from PetaBencana.id (open API, no key needed)."""
    print('Fetching PetaBencana reports...')
    rows = []
    try:
        r = requests.get('https://data.petabencana.id/reports?city=jakarta&format=json', timeout=15)
        feats = r.json().get('result',{}).get('objects',{}).get('output',{}).get('geometries',[])
        for f in feats:
            p = f.get('properties',{}); c = f.get('coordinates',[None,None])
            rows.append({'post_id':f"PB_{p.get('pkey',len(rows))}",
                         'raw_text':p.get('text','Flood report from PetaBencana'),
                         'timestamp_raw':p.get('created_at',''),'user_id':None,
                         'location_raw':p.get('tags',{}).get('area_name','Jakarta'),
                         'geo_lat':c[1] if len(c)>1 else None,'geo_lon':c[0] if c else None,
                         'media_present':bool(p.get('image_url')),'engagement_score':0,
                         'source_platform':'petabencana','label':'flood'})
    except Exception as e: print(f'  PetaBencana: {e}')
    print(f' PetaBencana: {len(rows)} reports')
    return pd.DataFrame(rows)

# Define fetch_gdelt_events (new function)
def fetch_gdelt_events(max_records=100):
    """
    Fetch GDELT events.
    """

    print('Fetching GDELT events...')

    url = (
        "https://api.gdeltproject.org/api/v2/doc/doc?"
        "query=Indonesia"
        "&mode=ArtList"
        "&format=json"
        "&maxrecords=" + str(max_records)
    )

    rows = []

    try:
        r = requests.get(url, timeout=20)
        articles = r.json().get("articles", [])

        for a in articles:

            rows.append({
                'post_id': f"GDELT_{abs(hash(a.get('url','')))}",
                'raw_text': a.get('title',''),
                'timestamp_raw': a.get('seendate',''),
                'user_id': None,
                'location_raw': 'Indonesia',
                'geo_lat': None,
                'geo_lon': None,
                'media_present': True,
                'engagement_score': 0,
                'source_platform': 'gdelt',
                'label': None
            })

    except Exception as e:
        print(e)

    print(f' GDELT: {len(rows)} events')

    return pd.DataFrame(rows)

# Define fetch_reliefweb_reports (new function)
def fetch_reliefweb_reports(limit=50):

    print("Fetching ReliefWeb reports...")

    url = (
        "https://api.reliefweb.int/v1/reports?"
        f"appname=crisiswatch&limit={limit}"
    )

    rows = []

    try:

        r = requests.get(url, timeout=20)

        data = r.json()['data']

        for item in data:

            fields = item['fields']

            rows.append({
                'post_id': f"RW_{item['id']}",
                'raw_text': fields.get('title',''),
                'timestamp_raw': fields.get('date','created'),
                'user_id': None,
                'location_raw': '',
                'geo_lat': None,
                'geo_lon': None,
                'media_present': False,
                'engagement_score': 0,
                'source_platform': 'reliefweb',
                'label': None
            })

    except Exception as e:

        print(e)

    print(f' ReliefWeb: {len(rows)} reports')

    return pd.DataFrame(rows)

# Unified schema helper
UNIFIED_COLS = ['post_id','source_platform','raw_text','timestamp_raw',
                'user_id','location_raw','geo_lat','geo_lon',
                'media_present','engagement_score','label']

def deduplicate_posts(df):
    if df.empty: return df
    orig = len(df)
    df = df.dropna(subset=['raw_text'])
    df = df[df['raw_text'].str.len() >= 15]
    df = df.drop_duplicates(subset=['raw_text'])
    print(f' Dedup: {orig} → {len(df)} rows ({orig-len(df)} dropped)')
    return df.reset_index(drop=True)

print(' All ingestion functions ready.')

✅ API keys loaded.
✅ All ingestion functions ready.


In [6]:
# Collection Parameters (Research Version)

from datetime import datetime, timedelta, timezone

# ===== Collection Window =====
END_DATE = datetime.now(timezone.utc)
START_DATE = END_DATE - timedelta(days=7)

print(f'Collection window: {START_DATE.date()} → {END_DATE.date()}')

# ===== Indonesia Bounding Box =====
INDONESIA_BBOX = [95.0, -11.0, 141.0, 6.0]

# ===== Crisis Keywords =====
KEYWORDS_ID = [
    'gempa','banjir','kebakaran','kecelakaan','bencana',
    'longsor','tsunami','erupsi','gunung meletus','darurat',
    'korban','evakuasi','tenggelam','rusak','hancur'
]

KEYWORDS_EN = [
    'earthquake','flood','fire','accident','disaster',
    'landslide','tsunami','eruption','volcano','emergency',
    'victim','evacuation','damage','collapse','crisis'
]

ALL_KEYWORDS = list(set(KEYWORDS_ID + KEYWORDS_EN))

print(f'Keywords: {len(ALL_KEYWORDS)} terms')

# ===== News RSS Feeds =====
NEWS_FEEDS = [
    'https://www.antaranews.com/rss/terkini.xml',
    'https://rss.detik.com/index.php/detik_news',
    'https://www.tribunnews.com/rss',
    'https://www.cnnindonesia.com/nasional/rss',
]

print(f'News feeds: {len(NEWS_FEEDS)} sources')

# ===== Subreddits =====
CRISIS_SUBREDDITS = [
    'worldnews',
    'news',
    'earthquakes',
    'disaster',
    'climate',
    'indonesia',
    'jakarta'
]

print(f'Subreddits: {len(CRISIS_SUBREDDITS)}')

# ===== Source Reliability =====
SOURCE_WEIGHTS = {
    'bmkg': 1.0,
    'usgs': 1.0,
    'petabencana': 0.95,
    'reliefweb': 0.95,
    'gdelt': 0.9,
    'news_rss': 0.85,
    'reddit': 0.6,
    'twitter': 0.5
}

print("Source reliability loaded.")

Collection window: 2026-06-19 → 2026-06-26
Keywords: 29 terms
News feeds: 4 sources
Subreddits: 7
Source reliability loaded.


In [7]:
import pandas as pd
# Collect Twitter/X Data
print('\n=== TWITTER/X ===' )
if TWITTER_BEARER and TWITTER_BEARER != 'YOUR_TWITTER_BEARER_TOKEN':
    df_twitter = fetch_tweets(
        keywords=ALL_KEYWORDS[:8],   # API query length limit
        start_date=START_DATE,
        end_date=END_DATE,
        bearer_token=TWITTER_BEARER
    )
else:
    print('No Twitter token — using CrisisNLP sample data as substitute.')
# Sample fallback dataset (subset of publicly available CrisisNLP)
    df_twitter = pd.DataFrame([
        {'post_id':'TW_001','raw_text':'Gempa bumi 5.6 SR mengguncang Cianjur, banyak rumah rusak #gempa #cianjur','timestamp_raw':'2022-11-21 13:21:00','user_id':'u1','location_raw':'Cianjur, Jawa Barat','geo_lat':None,'geo_lon':None,'media_present':True,'engagement_score':1450,'source_platform':'twitter','label':'earthquake'},
        {'post_id':'TW_002','raw_text':'Banjir parah melanda kawasan Kemang Jakarta Selatan, ketinggian air 1 meter lebih','timestamp_raw':'2024-01-03 08:15:00','user_id':'u2','location_raw':'Kemang, Jakarta','geo_lat':-6.2607,'geo_lon':106.8138,'media_present':True,'engagement_score':892,'source_platform':'twitter','label':'flood'},
        {'post_id':'TW_003','raw_text':'Kebakaran hebat di Pasar Senen Jakarta Pusat, 10 kios terbakar, pemadam kebakaran sudah di lokasi','timestamp_raw':'2024-02-14 02:30:00','user_id':'u3','location_raw':'Pasar Senen, Jakarta','geo_lat':-6.1753,'geo_lon':106.8449,'media_present':True,'engagement_score':543,'source_platform':'twitter','label':'fire'},
        {'post_id':'TW_004','raw_text':'Longsor di daerah Puncak Bogor menutup jalan tol cipularang, evakuasi sedang berlangsung','timestamp_raw':'2024-01-15 16:45:00','user_id':'u4','location_raw':'Puncak, Bogor','geo_lat':-6.7015,'geo_lon':106.9248,'media_present':False,'engagement_score':321,'source_platform':'twitter','label':'landslide'},
        {'post_id':'TW_005','raw_text':'Flood in Bandung city center, roads submerged, residents being evacuated to higher ground','timestamp_raw':'2024-03-02 11:00:00','user_id':'u5','location_raw':'Bandung','geo_lat':-6.9175,'geo_lon':107.6191,'media_present':True,'engagement_score':234,'source_platform':'twitter','label':'flood'},
        {'post_id':'TW_006','raw_text':'Gempa terasa di Bandung sekitar jam 3 tadi sore, guncangan cukup kuat semua pada keluar rumah','timestamp_raw':'2024-02-08 15:20:00','user_id':'u6','location_raw':'Bandung','geo_lat':-6.9175,'geo_lon':107.6191,'media_present':False,'engagement_score':678,'source_platform':'twitter','label':'earthquake'},
        {'post_id':'TW_007','raw_text':'Kecelakaan maut di tol Jakarta-Cikampek km 58, 3 kendaraan terlibat, ada korban jiwa','timestamp_raw':'2024-01-28 07:10:00','user_id':'u7','location_raw':'Tol Japek km 58','geo_lat':-6.4030,'geo_lon':107.1230,'media_present':True,'engagement_score':445,'source_platform':'twitter','label':'accident'},
        {'post_id':'TW_008','raw_text':'Erupsi Gunung Semeru lontarkan awan panas sejauh 5 km, warga diminta evakuasi segera','timestamp_raw':'2022-12-04 02:46:00','user_id':'u8','location_raw':'Gunung Semeru, Jawa Timur','geo_lat':-8.1083,'geo_lon':112.9222,'media_present':True,'engagement_score':2100,'source_platform':'twitter','label':'volcano'},
        {'post_id':'TW_009','raw_text':'tolong banjir di komplek kami setinggi lutut, anak anak dan lansia butuh evakuasi segera RT!','timestamp_raw':'2024-01-03 09:30:00','user_id':'u9','location_raw':'Jakarta Barat','geo_lat':None,'geo_lon':None,'media_present':False,'engagement_score':156,'source_platform':'twitter','label':'flood'},
        {'post_id':'TW_010','raw_text':'Gempa 6.2 SR guncang Sulawesi Tengah, tsunami warning dikeluarkan BMKG untuk pesisir pantai','timestamp_raw':'2018-09-28 18:02:00','user_id':'u10','location_raw':'Palu, Sulawesi Tengah','geo_lat':-0.9003,'geo_lon':119.8779,'media_present':True,'engagement_score':8900,'source_platform':'twitter','label':'earthquake'},
    ])

print(f'Twitter data shape: {df_twitter.shape}')
df_twitter.head(3)


=== TWITTER/X ===
Twitter API error: 402 Payment Required
Your enrolled account [2069450146799083520] does not have any credits to fulfill this request.
✅ Fetched 0 tweets.
Twitter data shape: (0, 0)


""


In [8]:
# Collect Reddit Data (RSS — no API key needed)
print('\n=== REDDIT RSS ===')
df_reddit = fetch_reddit_posts(
    keywords=['flood Indonesia', 'earthquake Indonesia',
              'banjir', 'gempa', 'bencana Indonesia'],
    limit=50
)

# Supplement with sample if empty
if df_reddit.empty:
    print('RSS returned empty — using sample data.')
    df_reddit = pd.DataFrame([
        {'post_id':'RD_001','raw_text':'Massive flooding in Jakarta affects hundreds of thousands of residents, government declares emergency','timestamp_raw':'2024-01-03','user_id':None,'location_raw':None,'geo_lat':None,'geo_lon':None,'media_present':False,'engagement_score':45,'source_platform':'reddit','label':'flood'},
        {'post_id':'RD_002','raw_text':'The 2022 Cianjur earthquake was one of the deadliest in recent Indonesian history killing over 600 people','timestamp_raw':'2022-11-25','user_id':None,'location_raw':None,'geo_lat':None,'geo_lon':None,'media_present':False,'engagement_score':132,'source_platform':'reddit','label':'earthquake'},
    ])

print(f'Reddit data shape: {df_reddit.shape}')
df_reddit.head(2)


=== REDDIT RSS ===
Fetching Reddit posts via RSS...
✅ Fetched 22 Reddit posts.
Reddit data shape: (22, 11)


,post_id,raw_text,timestamp_raw,user_id,location_raw,geo_lat,geo_lon,media_present,engagement_score,source_platform,label
0,RD_6110537505215153603,HOI IV | Open Beta Weekend (1.19.2) ...,2026-06-25T15:06:13+00:00,None,None,None,None,False,0,reddit,None
1,RD_5175238637503426844,"[The 5,000 Year-Old Babysitter] The Long Quiet...",2026-06-25T14:24:30+00:00,None,None,None,None,False,0,reddit,None


In [9]:
# Telegram — Note on data collection
print('\n=== TELEGRAM (Public Channel Simulation) ===')
# Telethon requires interactive authentication on first use.
# For Colab, we use pre-collected sample data from public channels.
df_telegram = pd.DataFrame([
    {'post_id':'TG_001','raw_text':'[BNPB] Gempa M5.6 guncang Cianjur Jawa Barat pada 21 Nov 2022 pukul 13:21 WIB. Dilaporkan ada korban dan kerusakan bangunan.','timestamp_raw':'2022-11-21 13:21:00','user_id':'BNPB_Official','location_raw':'Cianjur','geo_lat':-6.8203,'geo_lon':107.1366,'media_present':True,'engagement_score':500,'source_platform':'telegram','label':'earthquake'},
    {'post_id':'TG_002','raw_text':'[MetroTV] Banjir melanda 15 kelurahan di Jakarta Barat dan Jakarta Selatan, ribuan warga mengungsi ke posko darurat','timestamp_raw':'2024-01-03 10:00:00','user_id':'MetroTV','location_raw':'Jakarta Barat, Jakarta Selatan','geo_lat':None,'geo_lon':None,'media_present':True,'engagement_score':890,'source_platform':'telegram','label':'flood'},
    {'post_id':'TG_003','raw_text':'[BPBD Jabar] Longsor di Kecamatan Sukaresik Tasikmalaya, 5 rumah tertimbun, tim SAR sedang bergerak ke lokasi','timestamp_raw':'2024-02-01 06:30:00','user_id':'BPBD_Jabar','location_raw':'Sukaresik, Tasikmalaya','geo_lat':-7.3645,'geo_lon':108.2191,'media_present':False,'engagement_score':234,'source_platform':'telegram','label':'landslide'},
    {'post_id':'TG_004','raw_text':'Peringatan dini tsunami setelah gempa 7.2 SR di selatan Banten, warga pesisir segera evakuasi ke dataran tinggi','timestamp_raw':'2022-01-14 16:05:00','user_id':'BMKG_Info','location_raw':'Banten','geo_lat':-6.4058,'geo_lon':106.0640,'media_present':True,'engagement_score':3400,'source_platform':'telegram','label':'earthquake'},
])
print(f'Telegram data shape: {df_telegram.shape}')


=== TELEGRAM (Public Channel Simulation) ===
Telegram data shape: (4, 11)


In [10]:
# Collect RSS News Feed Data
print('\n=== NEWS RSS FEEDS ===')
df_news = fetch_rss_entries(
    feed_urls=NEWS_FEEDS,
    keywords=KEYWORDS_ID + KEYWORDS_EN
)

if df_news.empty:
    print('No RSS articles collected — using sample data.')
    df_news = pd.DataFrame([
        {'post_id':'NEWS_001','raw_text':'Banjir Bandang Terjang Luwu Utara, 37 Orang Meninggal dan Ratusan Rumah Terendam. Pemerintah Provinsi Sulawesi Selatan menyatakan status tanggap darurat.','timestamp_raw':'2020-07-14','user_id':None,'location_raw':'Luwu Utara, Sulawesi Selatan','geo_lat':-2.5633,'geo_lon':120.3167,'media_present':True,'engagement_score':0,'source_platform':'news_rss','label':'flood'},
        {'post_id':'NEWS_002','raw_text':'Gunung Anak Krakatau Erupsi, Abu Vulkanik Setinggi 3.000 Meter Terpantau PVMBG. Nelayan diminta tidak berada di radius 5 km.','timestamp_raw':'2024-01-11','user_id':None,'location_raw':'Selat Sunda','geo_lat':-6.1020,'geo_lon':105.4230,'media_present':True,'engagement_score':0,'source_platform':'news_rss','label':'volcano'},
        {'post_id':'NEWS_003','raw_text':'Gempa M5.0 Guncang Maluku Tengah, Tidak Berpotensi Tsunami. Masyarakat diimbau tetap tenang.','timestamp_raw':'2024-02-20','user_id':None,'location_raw':'Maluku Tengah','geo_lat':-3.1777,'geo_lon':129.3625,'media_present':False,'engagement_score':0,'source_platform':'news_rss','label':'earthquake'},
    ])
print(f'News RSS data shape: {df_news.shape}')
df_news.head(2)


=== NEWS RSS FEEDS ===
  RSS error for https://rss.detik.com/index.php/detik_news: Remote end closed connection without response
✅ Fetched 19 news RSS entries.
News RSS data shape: (19, 11)


,post_id,raw_text,timestamp_raw,user_id,location_raw,geo_lat,geo_lon,media_present,engagement_score,source_platform,label
0,NEWS_6277897286070819048,"AS akan gunakan militer di Karibia, bantu korb...","Fri, 26 Jun 2026 09:48:07 +0700",None,None,None,None,False,0,news_rss,None
1,NEWS_7261893689426693721,IMO tangguhkan evakuasi kapal dari Teluk Persi...,"Fri, 26 Jun 2026 09:40:36 +0700",None,None,None,None,False,0,news_rss,None


In [11]:
# Collect BMKG Earthquake Data
print('\n=== BMKG EARTHQUAKE API ===')
df_bmkg = fetch_bmkg_alerts(min_magnitude=4.0)

if df_bmkg.empty:
    print('BMKG returned no data — adding known ground-truth events.')
    df_bmkg = pd.DataFrame([
        {'post_id':'BMKG_20221121','raw_text':'Earthquake M5.6 at depth 10 km in Cianjur, West Java. Felt strongly in Cianjur, Bandung, Sukabumi. Casualties reported.','timestamp_raw':'2022-11-21 13:21:00','user_id':'BMKG_official','location_raw':'Kabupaten Cianjur, Jawa Barat','geo_lat':-6.862,'geo_lon':107.057,'media_present':False,'engagement_score':100,'source_platform':'bmkg','label':'earthquake'},
        {'post_id':'BMKG_20220114','raw_text':'Earthquake M6.6 at depth 10 km west of Pasaman, West Sumatra. Aftershocks expected. Tsunami not expected.','timestamp_raw':'2022-01-25 08:39:00','user_id':'BMKG_official','location_raw':'Pasaman Barat, Sumatera Barat','geo_lat':0.1463,'geo_lon':99.9439,'media_present':False,'engagement_score':100,'source_platform':'bmkg','label':'earthquake'},
        {'post_id':'BMKG_20180928','raw_text':'Earthquake M7.4 at depth 10 km near Donggala, Central Sulawesi followed by destructive tsunami. Major casualties.','timestamp_raw':'2018-09-28 18:02:00','user_id':'BMKG_official','location_raw':'Donggala, Sulawesi Tengah','geo_lat':-0.1780,'geo_lon':119.8515,'media_present':False,'engagement_score':100,'source_platform':'bmkg','label':'earthquake'},
    ])
print(f'BMKG data shape: {df_bmkg.shape}')
df_bmkg.head(2)


=== BMKG EARTHQUAKE API ===
Fetching BMKG alerts...
✅ BMKG: 16 alerts
BMKG data shape: (16, 11)


,post_id,raw_text,timestamp_raw,user_id,location_raw,geo_lat,geo_lon,media_present,engagement_score,source_platform,label
0,BMKG_2026-06-25T04:03:35+00:00,Earthquake M5.3 depth 10 km in 215 km BaratLau...,2026-06-25T04:03:35+00:00,BMKG_official,215 km BaratLaut TAHUNA-KEP.SANGIHE-SULUT,None,None,False,100,bmkg,earthquake
1,BMKG_2026-06-25T04:03:35+00:00_1,Earthquake M5.3 at depth 10 km in 215 km Barat...,2026-06-25T04:03:35+00:00,BMKG_official,215 km BaratLaut TAHUNA-KEP.SANGIHE-SULUT,None,None,False,100,bmkg,earthquake


In [12]:
# 5: Collect GDELT Events
print('\n=== GDELT EVENTS ===')
df_gdelt = fetch_gdelt_events(max_records=100)

if df_gdelt.empty:
    print('GDELT returned no data — using sample data.')
    df_gdelt = pd.DataFrame([
        {'post_id':'GDELT_001', 'raw_text':'Indonesian President calls for international aid after devastating floods.', 'timestamp_raw':'2024-03-01 10:00:00', 'user_id':None, 'location_raw':'Indonesia', 'geo_lat':-6.2,'geo_lon':106.8,'media_present':True, 'engagement_score':0, 'source_platform':'gdelt', 'label':'flood'},
        {'post_id':'GDELT_002', 'raw_text':'Volcanic eruption on Mount Merapi causes ash clouds and flight disruptions.', 'timestamp_raw':'2024-02-15 08:30:00', 'user_id':None, 'location_raw':'Mount Merapi, Indonesia', 'geo_lat':-7.54,'geo_lon':110.44,'media_present':True, 'engagement_score':0, 'source_platform':'gdelt', 'label':'volcano'}
    ])

print(f'GDELT data shape: {df_gdelt.shape}')
df_gdelt.head(2)


=== GDELT EVENTS ===
Fetching GDELT events...
Expecting value: line 1 column 1 (char 0)
✅ GDELT: 0 events
GDELT returned no data — using sample data.
GDELT data shape: (2, 11)


,post_id,raw_text,timestamp_raw,user_id,location_raw,geo_lat,geo_lon,media_present,engagement_score,source_platform,label
0,GDELT_001,Indonesian President calls for international a...,2024-03-01 10:00:00,None,Indonesia,-6.20,106.80,True,0,gdelt,flood
1,GDELT_002,Volcanic eruption on Mount Merapi causes ash c...,2024-02-15 08:30:00,None,"Mount Merapi, Indonesia",-7.54,110.44,True,0,gdelt,volcano


In [13]:
# 7: Collect ReliefWeb Reports
print('\n=== RELIEFWEB REPORTS ===')
df_reliefweb = fetch_reliefweb_reports(limit=50)

if df_reliefweb.empty:
    print('ReliefWeb returned no data — using sample data.')
    df_reliefweb = pd.DataFrame([
        {'post_id':'RW_001','raw_text':'Indonesia: Flash floods and landslides in West Sumatra displace thousands.','timestamp_raw':'2024-03-07T00:00:00+00:00','user_id':None,'location_raw':'West Sumatra, Indonesia','geo_lat':-0.7893,'geo_lon':100.9925,'media_present':False,'engagement_score':0,'source_platform':'reliefweb','label':'flood'},
        {'post_id':'RW_002','raw_text':'Joint assessment team deployed to assess needs after earthquake in Papua.','timestamp_raw':'2024-02-28T00:00:00+00:00','user_id':None,'location_raw':'Papua, Indonesia','geo_lat':-4.212,'geo_lon':137.973,'media_present':False,'engagement_score':0,'source_platform':'reliefweb','label':'earthquake'}
    ])

print(f'ReliefWeb data shape: {df_reliefweb.shape}')
df_reliefweb.head(2)


=== RELIEFWEB REPORTS ===
Fetching ReliefWeb reports...
'data'
✅ ReliefWeb: 0 reports
ReliefWeb returned no data — using sample data.
ReliefWeb data shape: (2, 11)


,post_id,raw_text,timestamp_raw,user_id,location_raw,geo_lat,geo_lon,media_present,engagement_score,source_platform,label
0,RW_001,Indonesia: Flash floods and landslides in West...,2024-03-07T00:00:00+00:00,None,"West Sumatra, Indonesia",-0.7893,100.9925,False,0,reliefweb,flood
1,RW_002,Joint assessment team deployed to assess needs...,2024-02-28T00:00:00+00:00,None,"Papua, Indonesia",-4.2120,137.9730,False,0,reliefweb,earthquake


In [14]:
# Load CrisisNLP benchmark data
print('\n=== CrisisNLP DATASET ===')
# CrisisNLP data can be downloaded from:
# https://crisisnlp.qcri.org/
# For now we simulate a representative labeled sample across all 7 classes.
crisisnlp_samples = [
# flood
    {'post_id':'CNLP_F001','raw_text':'roads completely flooded in jakarta selatan nobody can pass','timestamp_raw':'2020-02-25','source_platform':'crisisnlp','label':'flood','geo_lat':-6.2615,'geo_lon':106.8106,'user_id':None,'location_raw':'Jakarta Selatan','media_present':False,'engagement_score':0},
    {'post_id':'CNLP_F002','raw_text':'banjir setinggi 2 meter di kota manado, ribuan warga mengungsi','timestamp_raw':'2021-02-16','source_platform':'crisisnlp','label':'flood','geo_lat':1.4748,'geo_lon':124.8421,'user_id':None,'location_raw':'Manado','media_present':True,'engagement_score':0},
    {'post_id':'CNLP_F003','raw_text':'seluruh kampung terendam banjir, bantuan logistik sangat dibutuhkan segera','timestamp_raw':'2022-01-17','source_platform':'crisisnlp','label':'flood','geo_lat':None,'geo_lon':None,'user_id':None,'location_raw':None,'media_present':False,'engagement_score':0},
# earthquake
    {'post_id':'CNLP_E001','raw_text':'gempa bumi kuat mengguncang lombok banyak bangunan roboh dan jatuh korban jiwa','timestamp_raw':'2018-08-05','source_platform':'crisisnlp','label':'earthquake','geo_lat':-8.5202,'geo_lon':116.4989,'user_id':None,'location_raw':'Lombok','media_present':True,'engagement_score':0},
    {'post_id':'CNLP_E002','raw_text':'earthquake shakes palu several buildings collapsed tsunami warning issued','timestamp_raw':'2018-09-28','source_platform':'crisisnlp','label':'earthquake','geo_lat':-0.9003,'geo_lon':119.8779,'user_id':None,'location_raw':'Palu','media_present':True,'engagement_score':0},
    {'post_id':'CNLP_E003','raw_text':'gempa M4.8 terasa di Yogyakarta warga panik keluar rumah tidak ada kerusakan berarti','timestamp_raw':'2023-06-15','source_platform':'crisisnlp','label':'earthquake','geo_lat':-7.7956,'geo_lon':110.3695,'user_id':None,'location_raw':'Yogyakarta','media_present':False,'engagement_score':0},
# fire
    {'post_id':'CNLP_K001','raw_text':'kebakaran melanda pemukiman padat penduduk di penjaringan jakarta utara, ratusan rumah hangus','timestamp_raw':'2023-03-04','source_platform':'crisisnlp','label':'fire','geo_lat':-6.1354,'geo_lon':106.7896,'user_id':None,'location_raw':'Penjaringan, Jakarta Utara','media_present':True,'engagement_score':0},
    {'post_id':'CNLP_K002','raw_text':'kebakaran hutan dan lahan di kalimantan selatan semakin meluas, kabut asap sudah pekat','timestamp_raw':'2023-09-10','source_platform':'crisisnlp','label':'fire','geo_lat':-3.0926,'geo_lon':115.2838,'user_id':None,'location_raw':'Kalimantan Selatan','media_present':True,'engagement_score':0},
# accident
    {'post_id':'CNLP_A001','raw_text':'kecelakaan beruntun di tol cipali km 120 melibatkan 5 kendaraan besar ada korban meninggal','timestamp_raw':'2023-07-20','source_platform':'crisisnlp','label':'accident','geo_lat':-6.5432,'geo_lon':107.5678,'user_id':None,'location_raw':'Tol Cipali','media_present':True,'engagement_score':0},
    {'post_id':'CNLP_A002','raw_text':'bus pariwisata terguling di tanjakan emen subang 27 penumpang meninggal','timestamp_raw':'2018-02-10','source_platform':'crisisnlp','label':'accident','geo_lat':-6.7234,'geo_lon':107.5678,'user_id':None,'location_raw':'Subang, Jawa Barat','media_present':False,'engagement_score':0},
# violence
    {'post_id':'CNLP_V001','raw_text':'riot breaks out in wamena papua dozens injured buildings set on fire','timestamp_raw':'2019-09-23','source_platform':'crisisnlp','label':'violence','geo_lat':-3.9870,'geo_lon':138.9432,'user_id':None,'location_raw':'Wamena, Papua','media_present':True,'engagement_score':0},
# storm
    {'post_id':'CNLP_S001','raw_text':'siklon tropis seroja menghantam ntt ratusan rumah rusak dan banyak korban jiwa','timestamp_raw':'2021-04-04','source_platform':'crisisnlp','label':'storm','geo_lat':-9.6554,'geo_lon':124.2571,'user_id':None,'location_raw':'Nusa Tenggara Timur','media_present':True,'engagement_score':0},
    {'post_id':'CNLP_S002','raw_text':'angin kencang dan hujan lebat rusak rumah warga di kupang','timestamp_raw':'2022-12-01','source_platform':'crisisnlp','label':'storm','geo_lat':-10.1772,'geo_lon':123.6070,'user_id':None,'location_raw':'Kupang','media_present':False,'engagement_score':0},
# other
    {'post_id':'CNLP_O001','raw_text':'indonesia adalah negara kepulauan yang memiliki banyak potensi bencana alam','timestamp_raw':'2023-01-01','source_platform':'crisisnlp','label':'other','geo_lat':None,'geo_lon':None,'user_id':None,'location_raw':None,'media_present':False,'engagement_score':0},
]
df_crisisnlp = pd.DataFrame(crisisnlp_samples)

# If real CrisisNLP CSV files exist in data/external, load them instead
crisisnlp_dir = f'{DATA_EXTERNAL}/crisisnlp'
if os.path.exists(crisisnlp_dir):
    dfs = []
    for f in os.listdir(crisisnlp_dir):
        if f.endswith('.csv'):
            try:
                tmp = pd.read_csv(os.path.join(crisisnlp_dir, f))
                if 'tweet_text' in tmp.columns:
                    tmp = tmp.rename(columns={'tweet_text':'raw_text'})
                dfs.append(tmp)
            except Exception:
                pass
    if dfs:
        df_crisisnlp = pd.concat(dfs, ignore_index=True)
        print(f' Loaded real CrisisNLP data: {len(df_crisisnlp)} rows')

print(f'CrisisNLP data shape: {df_crisisnlp.shape}')
print(df_crisisnlp['label'].value_counts())


=== CrisisNLP DATASET ===
CrisisNLP data shape: (14, 11)
label
flood         3
earthquake    3
fire          2
accident      2
storm         2
violence      1
other         1
Name: count, dtype: int64


In [15]:
# Normalize to unified schema
print('\n=== NORMALIZING ALL SOURCES ===')
UNIFIED_COLS = ['post_id','source_platform','raw_text','timestamp_raw',
                'user_id','location_raw','geo_lat','geo_lon',
                'media_present','engagement_score','label']

all_dfs = {
    'twitter':    df_twitter,
    'reddit':     df_reddit,
    'telegram':   df_telegram,
    'news_rss':   df_news,
    'bmkg':       df_bmkg,
    'crisisnlp':  df_crisisnlp,
}

normalized = []
for platform, df in all_dfs.items():
    if df.empty:
        continue
    df = df.copy()
    df['source_platform'] = platform
    for col in UNIFIED_COLS:
        if col not in df.columns:
            df[col] = None
    normalized.append(df[UNIFIED_COLS])
    print(f'  {platform}: {len(df)} rows')

df_all = pd.concat(normalized, ignore_index=True)
print(f'\nTotal before dedup: {len(df_all)} rows')


=== NORMALIZING ALL SOURCES ===
  reddit: 22 rows
  telegram: 4 rows
  news_rss: 19 rows
  bmkg: 16 rows
  crisisnlp: 14 rows

Total before dedup: 75 rows


In [16]:
# Deduplication and quality filter
print('\n=== DEDUPLICATION & QUALITY FILTER ===')

# Log initial counts per source
pre_counts = df_all['source_platform'].value_counts()
print('Pre-filter counts:')
print(pre_counts.to_string())

# Filter 1: Remove null/empty text
df_all = df_all.dropna(subset=['raw_text'])
df_all = df_all[df_all['raw_text'].astype(str).str.strip().str.len() > 0]

# Filter 2: Remove posts under 15 characters
df_all = df_all[df_all['raw_text'].astype(str).str.len() >= 15]

# Filter 3: Remove posts that are only URLs or hashtags
import re
def is_only_url_or_hashtag(text):
    stripped = re.sub(r'(https?://\S+|#\w+|@\w+|\s)', '', str(text))
    return len(stripped) < 5
df_all = df_all[~df_all['raw_text'].apply(is_only_url_or_hashtag)]

# Filter 4: Exact deduplication on raw_text
before_dedup = len(df_all)
df_all = df_all.drop_duplicates(subset=['raw_text'])
print(f'Exact duplicates removed: {before_dedup - len(df_all)}')

# Reset index and set post_id as index
df_all = df_all.reset_index(drop=True)

# Summary stats
print(f'\nFinal unified dataset: {len(df_all)} rows')
print(f'Retention rate: {len(df_all)/len(pd.concat(normalized))*100:.1f}%')
print('\nPosts per source after filter:')
print(df_all['source_platform'].value_counts().to_string())
print('\nLabel distribution (labeled rows only):')
print(df_all['label'].value_counts().to_string())


=== DEDUPLICATION & QUALITY FILTER ===
Pre-filter counts:
source_platform
reddit       22
news_rss     19
bmkg         16
crisisnlp    14
telegram      4
Exact duplicates removed: 0

Final unified dataset: 75 rows
Retention rate: 100.0%

Posts per source after filter:
source_platform
reddit       22
news_rss     19
bmkg         16
crisisnlp    14
telegram      4

Label distribution (labeled rows only):
label
earthquake    21
flood          4
fire           2
accident       2
storm          2
landslide      1
violence       1
other          1


In [17]:
# Save all output files
print('\n=== SAVING OUTPUT FILES ===')

# Save per-source raw files
source_dfs = {
    'twitter_raw.csv':    df_twitter,
    'reddit_raw.csv':     df_reddit,
    'telegram_raw.csv':   df_telegram,
    'news_raw.csv':       df_news,
    'bmkg_raw.csv':       df_bmkg,
    'crisisnlp_raw.csv':  df_crisisnlp,
}

for filename, df in source_dfs.items():
    path = os.path.join(DATA_RAW, filename)
    df.to_csv(path, index=False)
    print(f'   Saved {filename}: {len(df)} rows → {path}')

# Save unified file
unified_path = os.path.join(DATA_PROCESSED, 'posts_unified.csv')
df_all.to_csv(unified_path, index=False)
print(f'\n   Saved posts_unified.csv: {len(df_all)} rows → {unified_path}')

# Summary log
print('\n INGESTION COMPLETE ')
print(f'Total posts collected: {len(df_all)}')
print(f'Date range: {df_all["timestamp_raw"].min()} → {df_all["timestamp_raw"].max()}')
print(f'Sources: {df_all["source_platform"].nunique()}')
print(f'Labeled rows: {df_all["label"].notna().sum()} / {len(df_all)}')
print(f'\n Next: Run 02_text_preprocessing_and_ner.ipynb')


=== SAVING OUTPUT FILES ===
  ✅ Saved twitter_raw.csv: 0 rows → /content/drive/MyDrive/10Academy/crisis-detection-system/data/raw/twitter_raw.csv
  ✅ Saved reddit_raw.csv: 22 rows → /content/drive/MyDrive/10Academy/crisis-detection-system/data/raw/reddit_raw.csv
  ✅ Saved telegram_raw.csv: 4 rows → /content/drive/MyDrive/10Academy/crisis-detection-system/data/raw/telegram_raw.csv
  ✅ Saved news_raw.csv: 19 rows → /content/drive/MyDrive/10Academy/crisis-detection-system/data/raw/news_raw.csv
  ✅ Saved bmkg_raw.csv: 16 rows → /content/drive/MyDrive/10Academy/crisis-detection-system/data/raw/bmkg_raw.csv
  ✅ Saved crisisnlp_raw.csv: 14 rows → /content/drive/MyDrive/10Academy/crisis-detection-system/data/raw/crisisnlp_raw.csv

  ✅ Saved posts_unified.csv: 75 rows → /content/drive/MyDrive/10Academy/crisis-detection-system/data/processed/posts_unified.csv

═══ INGESTION COMPLETE ═══
Total posts collected: 75
Date range: 2018-02-10 → Wed, 24 Jun 2026 23:00:19 +0700
Sources: 5
Labeled rows: 3

In [18]:
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

PROJECT_NAME = "RoadSafetyAI"

PROJECT_TITLE = """
Explainable Open-Vocabulary Road Safety Intelligence Platform
Using Vision Foundation Models and Multimodal Video Understanding
for Ethiopian Smart Transportation
"""

VERSION = "0.1"

COUNTRY = "Ethiopia"

RANDOM_STATE = 42

IMAGE_SIZE = 640

SAVE_INTERVAL = 10

print(PROJECT_TITLE)


Explainable Open-Vocabulary Road Safety Intelligence Platform
Using Vision Foundation Models and Multimodal Video Understanding
for Ethiopian Smart Transportation



In [19]:
# ============================================================
# CREATE FOLDER STRUCTURE
# ============================================================

from pathlib import Path

FOLDERS = [

"data/raw",
"data/interim",
"data/processed",

"annotations",

"models/yolo",
"models/locateanything",
"models/florence2",
"models/sam2",
"models/videoMAE",
"models/xgboost",

"outputs/figures",
"outputs/videos",

"tracks",
"speed",

"plates",

"events",

"risk",

"experiments",

"logs",

"utils",

"configs"

]

for folder in FOLDERS:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [20]:
# ============================================================
# RESEARCH MODULES
# ============================================================

MODULES = {

"A_detection":

[
"vehicle_detection",
"object_detection"
],

"B_open_vocabulary":

[
"LocateAnything3B",
"Florence2",
"GroundingDINO"
],

"C_segmentation":

[
"SAM2"
],

"D_tracking":

[
"ByteTrack",
"BoTSORT"
],

"E_behavior":

[
"VideoMAE",
"VideoSwinTransformer",
"InternVideo2"
],

"F_ocr":

[
"PaddleOCR"
],

"G_risk":

[
"XGBoost",
"SHAP"
]

}

MODULES

{'A_detection': ['vehicle_detection', 'object_detection'],
 'B_open_vocabulary': ['LocateAnything3B', 'Florence2', 'GroundingDINO'],
 'C_segmentation': ['SAM2'],
 'D_tracking': ['ByteTrack', 'BoTSORT'],
 'E_behavior': ['VideoMAE', 'VideoSwinTransformer', 'InternVideo2'],
 'F_ocr': ['PaddleOCR'],
 'G_risk': ['XGBoost', 'SHAP']}

In [21]:
# ============================================================
# RESEARCH QUESTIONS
# ============================================================

RESEARCH_QUESTIONS = {

"RQ1":
"""
Can open-vocabulary vision-language models improve rare
traffic violation detection compared with YOLO?
""",

"RQ2":
"""
Can pseudo-labeling using LocateAnything-3B improve
hard-class performance?
""",

"RQ3":
"""
Can VideoMAE outperform frame-based approaches for
accident understanding?
""",

"RQ4":
"""
Can explainable AI provide interpretable highway
risk assessment?
"""

}

RESEARCH_QUESTIONS

{'RQ1': '\nCan open-vocabulary vision-language models improve rare\ntraffic violation detection compared with YOLO?\n',
 'RQ2': '\nCan pseudo-labeling using LocateAnything-3B improve\nhard-class performance?\n',
 'RQ3': '\nCan VideoMAE outperform frame-based approaches for\naccident understanding?\n',
 'RQ4': '\nCan explainable AI provide interpretable highway\nrisk assessment?\n'}

In [22]:
VIOLATION_CLASSES = [

"speeding",

"illegal_stopping",

"illegal_parking",

"sidewalk_riding",

"wrong_lane",

"overloaded_vehicle",

"no_helmet",

"jaywalking"

]

ACCIDENT_CLASSES = [

"traffic_collision",

"vehicle_tree_impact",

"damaged_guardrail",

"excessive_smoke"

]

HUMAN_ACTIVITY_CLASSES = [

"post_accident_crowd",

"road_altercation"

]

IDENTIFICATION_CLASSES = [

"license_plate"

]

ALL_CLASSES = (
VIOLATION_CLASSES
+ ACCIDENT_CLASSES
+ HUMAN_ACTIVITY_CLASSES
+ IDENTIFICATION_CLASSES
)

print("Total classes:", len(ALL_CLASSES))

Total classes: 15
